# Politician tweets · Concept enrichment and prediction

A self-contained demonstration of concept search, one-vs-rest party enrichment, and sparse L1 classification on a balanced sample of politician tweets. Results are exploratory; author leakage can inflate tweet-level classifier accuracy.

## Install ConceptExplorer from GitHub

Install `uce-concept-explorer` from `kennylpeng/ConceptExplorer` on GitHub. Use a release tag instead of `main` for stable results.


In [ ]:
%pip -q install "git+https://github.com/kennylpeng/ConceptExplorer.git@main"

from uce_concept_explorer import ConceptExplorer, load_metadata


## 1. Install and load UCE-Gemma

Select **Runtime → Change runtime type → T4 GPU** in Colab. This notebook downloads the model weights only for encoding. Subsequent analysis is CPU-only.

Colab already includes the notebook dependencies listed in its [GPU runtime package list](https://github.com/googlecolab/backend-info/blob/main/pip-freeze.gpu.txt). The next cell installs the model's requirements to satisfy its version constraints, including its pinned `transformers` version.

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = "klpeng/uce-gemma-80k"
model_dir = snapshot_download(MODEL_ID, token=False)

%pip -q install -r "{model_dir}/requirements.txt"


In [ ]:
import sys, torch, numpy as np
from scipy import sparse
sys.path.insert(0, model_dir)
from uce_gemma import UCEGemma
model = UCEGemma.from_pretrained(model_dir, device='cuda' if torch.cuda.is_available() else 'cpu')
print('Model loaded.')

## Load tweets

Uses the public [PoliticalTweets](https://huggingface.co/datasets/Jacobvs/PoliticalTweets) dataset. The cell prints the available columns before sampling.

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset('Jacobvs/PoliticalTweets', split='train', token=False)
print(ds.column_names)
df = ds.to_pandas()
assert {'text','party'} <= set(df.columns), 'Dataset field names changed; inspect columns above.'
df = df[df.text.map(lambda x: isinstance(x,str) and len(x.strip())>=30)]
df = df[df.party.isin(['Democrat','Republican'])]
n_per = min(500, *df.party.value_counts().reindex(['Democrat','Republican'], fill_value=0).tolist())
assert n_per > 0, 'No matching party rows; inspect df.party.value_counts()'
sample = pd.concat([g.sample(n=n_per,random_state=42) for _,g in df.groupby('party')]).sample(frac=1,random_state=42).reset_index(drop=True)
texts = sample.text.tolist()
labels = sample.party.to_numpy()
print(sample.party.value_counts())

## 2. Encode once

Batch size is deliberately small for a 16 GB T4. Cache the sparse scores to avoid rerunning the GPU. Documents are encoded with the same ordering as `texts`.

In [ ]:
from tqdm.auto import tqdm

THRESHOLD = 0.92
BATCH_SIZE = 2
chunks = []
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Encoding", unit="batch"):
    with torch.inference_mode():
        chunk = model.encode(list(texts[i:i+BATCH_SIZE]))
    chunks.append(sparse.csr_matrix(chunk))
X = sparse.vstack(chunks, format='csr')
sparse.save_npz('uce_scores.npz', X)
print('Scores:', X.shape, 'nnz:', X.nnz)
# Later sessions: X = sparse.load_npz('uce_scores.npz')

## Explore concepts (CPU only)

The implementation is imported from GitHub; only dataset-specific analysis appears here. The binary threshold applies to all analyses.


In [ ]:
# Metadata-only loader: no weights are downloaded by this call.
metadata = load_metadata(model_dir)
E = ConceptExplorer(X, texts, metadata, threshold=THRESHOLD)
print('Encoded texts:', len(texts), 'active features:', int((E.counts>0).sum()))
display(E.prevalent(top_n=15))

**Optional fidelity filtering.** The UCE model card describes F1 based on up to 30 positive examples and 30 sparse zero examples per concept; it is an *automated judgment* metric, not corpus-calibrated precision or recall.

In [ ]:
print('Fidelity columns:', [c for c in E.table.columns if c in ('f1','precision','recall','accuracy')])
# Uncomment if available:
# E.set_fidelity_filter(min_fidelity=0.8, field='f1')
# display(E.prevalent(top_n=10))

## Discover distinguishing concepts

For between-category enrichment, the comparison uses only this dataset. Corpus enrichment uses the UCE package's reference prevalence metadata and, when necessary, an explicit calibrated threshold mapping.


In [ ]:
display(E.search('democracy'))
for party, table in E.by_category(labels, top_n=15, multiplicative=False).items():
    print('Enriched in', party)
    display(table)
display(E.predictive(labels, kind='categorical', top_n=20))

In [ ]:
# Inspect original tweets for a selected concept ID
concept_id = int(E.prevalent(top_n=1).iloc[0].concept_id)
display(E.examples(concept_id, n=10))

In [ ]:
# Optional L1 model; random tweet-level split may share politicians across train/test.
clf, metrics, selected = E.lasso(labels, kind='categorical', top_n=20)
print(metrics)
display(selected)

## Fixed-budget prediction (new API)

Select up to five non-rare concepts jointly. This example uses tweet-level holdout; if author IDs are available, pass them as `groups` for better generalization assessment.


In [ ]:
clf, metrics, selected = E.classifier(labels, n_concepts=5, min_count=10)
print(metrics)
display(selected)

# To score new texts encoded with the *same concept vocabulary and threshold*:
# clf.predict(E.Z[:, E.allowed])
